# Dynamic models — pick the model per request, not per agent
**by Kassouar Fatima 🇩🇿 — Module 3 project**

`create_agent` takes **one** model — but a `@wrap_model_call` middleware sees the whole request and can swap the model before the handler runs. A **short** conversation answers on the cheap standard model; past 10 messages the **same agent** switches to a bigger model — and `model_name` proves which one answered.

- **Run:** double-click `run_3_4.cmd` (opens all three `3_4` notebooks) → *Run → Run All Cells*.
- **Needs:** `GOOGLE_API_KEY` from the `.env` next to this notebook.
- **Models:** standard `gemini-3.5-flash-lite` · long `gemini-3.8-flash` — two free buckets. **Model fix:** the course switched `claude-sonnet-4-6`/`gpt-5-nano` (two keys) → both Gemini here.

**Idea:** the model is just a field of the request — the next two notebooks override the prompt and the tools with exactly the same mechanism.

**What I learned:**
- `request.override(model=...)` then `handler(request)` — forgetting it keeps the old model. The choice can read `messages`, `state`, `runtime.context`.
- `response_metadata["model_name"]` is the receipt — the only honest check that your branch fired.
- Each model has its own 500/day bucket: per-request choice also spreads the load across budgets.


## Setup

In [1]:
import importlib.util as _ilu
import subprocess as _sp
import sys as _sys

# Install only what is missing. Uses pip when available, falls back to uv
# (the local venv on this machine is created by uv and has no pip).
REQUIRED = {
    "langchain": "langchain",
    "langgraph": "langgraph",
    "langchain_google_genai": "langchain-google-genai",
    "langchain_community": "langchain-community",
    "dotenv": "python-dotenv",
}
missing = [pkg for imp, pkg in REQUIRED.items() if _ilu.find_spec(imp) is None]

if missing:
    print("Installing:", ", ".join(missing))
    if _sp.run([_sys.executable, "-m", "pip", "install", "-q", *missing]).returncode != 0:
        _sp.run(["uv", "pip", "install", *missing])
else:
    print("All packages already installed.")

All packages already installed.


In [2]:
import os
from getpass import getpass
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))  # local run: finds the .env above the notebook folder

# Colab run: read the key from the Secrets panel (key icon in the left sidebar)
try:
    from google.colab import userdata
    try:
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except Exception:
        pass
except ImportError:
    pass

def is_real(value):
    # empty values and template placeholders ("your_..." ) count as missing
    return bool(value) and not value.startswith("your_")

# Accept both names: the local project keeps the key as GEMINI_API_KEY
for name in ("GOOGLE_API_KEY", "GEMINI_API_KEY"):
    if is_real(os.getenv(name)):
        os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"] = os.getenv(name)
        break

# Still missing? Paste it here (hidden input).
if not is_real(os.getenv("GOOGLE_API_KEY")):
    try:
        value = getpass("Gemini API key is missing - paste it (Enter to skip): ").strip()
    except Exception:
        value = ""
    if value:
        os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"] = value

print("GOOGLE_API_KEY:", "set" if is_real(os.getenv("GOOGLE_API_KEY")) else "MISSING")

# Tracing with a placeholder LangSmith key only produces 401/403 warnings: switch it off
if not is_real(os.getenv("LANGSMITH_API_KEY")):
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing: off (no valid LANGSMITH_API_KEY)")

GOOGLE_API_KEY: set
LangSmith tracing: off (no valid LANGSMITH_API_KEY)


In [3]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model = init_chat_model("google_genai:gemini-3.8-flash")
standard_model = init_chat_model("google_genai:gemini-3.5-flash-lite")


@wrap_model_call
def state_based_model(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)  

    return handler(request)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

[{'type': 'text', 'text': "Ah, thanks for reminding me! I got caught up making copies, but I'll go grab the watering can right now. \n\nIs it just the big fern in the breakroom, or did you want me to check the little succulents on reception too?", 'extras': {'signature': 'EmAKXgFpFH0TTQW4CmksiihoXYNhtS9x+nVVB1RZHyNFGn8JG8OfCXPvttA6ohUx4nPdfi0BYtd1UiTlY7hk8VVRwRekU4wp3NzpD7Jw9Wbuih23hWUrkFEXEsk2gAJJ91w='}}]


In [6]:
print(response["messages"][-1].response_metadata["model_name"])

gemini-3.5-flash-lite


In [7]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

[{'type': 'text', 'text': 'Probably in about a year or two, or whenever we start seeing roots poking out of the drainage holes at the bottom.', 'extras': {'signature': 'Eu4OCusOAWkUfRMZzWHR9dirQ06YpcBe/lPDTTA2KEZVClXSgsNzPG0S2itTJJbF5dszcq2b6SCcJzx/ltVsSnE7pLWqKlEaYfCbmApqKI5gcRXN3ccWENSnxwUJ37NK7ca1vfjMqORkecGGmeLZDX8cUYiwjKi7wq+rFxfZ1X58yti7QqSxApTV4XLA2PbDZeydQBn7nME8OhdTylEnEN5WST0WEJBLFaU5tD15oV9YHSQL8bz5QOrGyWqqgi5DuxC61/2gsE6LHkzgZYl2NGgvSzbJ2faoowfs4GiAh/E9JsHDLz0HvflVkdpLCHXvqrm2GLUFiqRuKUu3QJEr5eZeB/8uUfWNEMKd9V8Yr5HlmBbPTRIO1Gqr2TATyLLiZOnUd4L1klhoM4S/GoTWZjqi4mihB0S4t1T8TLP/ErpfV5wllGsQPrAcypvrliRAAwrWPGE9DR/9qNBqLYS4wnquA+65q3CXqrw+84Kziq1m+5KNmT50VQ10Lff5y9kZ3PgltrLaU5I/rbkNL4QWInRXpfAhlKFxzbeULwCHcGuq012mY5xqMBfo+EsyJild8aple1VHTlnImgYv6YAZVXPzSxQkdwCYPx0/RUbEoS8lroGxD+MWDpaeQLXG9/i/MsDnzcNaSCjk6fe7/07N0x9xerAh014/PbAm2gQmvcxyUjazEjTh2auFEwkmxeGYZnTajX4o1rofIYFrNBo1N+XAKz1tAZTY6mqI7xdqkNKT2gLCRYPO3w2QiFZzjfIsPk8E6dtgzZe4Ver3xvo3AbJF/1EyTgw23xY7MO9lUPFxG2jQt/t5i0x4HF255Dw

In [8]:
print(response["messages"][-1].response_metadata["model_name"])

gemini-3.8-flash


## Your task

> - Make one prompt dynamic. Anything: language, tone, level of detail. Get it switching on something real.
> - Hide a tool from one kind of user. This is the pattern behind every permission you will ever implement.
> - Then try the model switch, and put the token counts side by side. That is the difference between an agent that works and an agent that can afford to run.

### My response — by Kassouar Fatima 🇩🇿 — Module 3 project

This notebook does **task 3**; tasks 1 and 2 live in the two sibling notebooks
(`3_4_Dynamic_prompts_kassouar`, `3_4_Dynamic_tools_kassouar`).

**The switch.** `state_based_model` is a `@wrap_model_call` middleware: it reads
`len(request.messages)` (a shortcut for `request.state["messages"]`), picks `large_model` above 10
messages and `standard_model` below, then `request.override(model=model)` and `handler(request)`.
The agent is still *built* with one model — the middleware swaps it per call. The receipts, printed
by `response["messages"][-1].response_metadata["model_name"]`:

- 1 message → **`gemini-3.5-flash-lite`**
- 13 messages → **`gemini-3.8-flash`**

**Token counts side by side** (same prompts sent to both models, read from `usage_metadata`):

| conversation | 📈 `gemini-3.5-flash-lite` | 📈 `gemini-3.8-flash` |
|---|---|---|
| short — 1 message | 9 in / 52 out / **61 total** | 9 in / 227 out / **236 total** (181 of them reasoning) |
| long — 13 messages | 109 in / 44 out / **153 total** | 109 in / 433 out / **542 total** (412 of them reasoning) |

The **input is identical** on both sides (9, then 109) — the difference is entirely on the output
side. The cheap model answers in 52 and 44 tokens; the big one spends **181 and 412 tokens
reasoning first**, which is 236 vs 61 total (about ×4) on the short turn and 542 vs 153 (about
×3.5) on the long one.

**That is the difference between an agent that works and one that can afford to run.** The big
model is not wrong — it is a bigger context window and more thinking, and for a 13-message
conversation it is worth it. But you cannot pay ×4 on every one-line question in the thread. The
middleware routes the cheap turns to the cheap model and only the long ones to the expensive one,
so the bill follows the hard questions instead of every question.

## Summary — main functions & parameters

| Function / parameter | What it does |
|---|---|
| `init_chat_model("google_genai:gemini-3.5-flash-lite")` | `standard_model` — used for short conversations |
| `init_chat_model("google_genai:gemini-3.8-flash")` | `large_model` — used past the threshold (bigger window, more thinking) |
| `@wrap_model_call state_based_model(request, handler)` | sees the whole request before the model call and may swap it |
| `len(request.messages) > 10` | the switch condition — `request.messages` is a shortcut for `request.state["messages"]` |
| `request.override(model=model)` | returns a copy of the request with the new model; then `handler(request)` |
| `create_agent(model=..., middleware=[state_based_model])` | the agent declares a **default** model; the middleware overrides it per call |
| `response["messages"][-1].response_metadata["model_name"]` | the receipt — which model actually answered (`gemini-3.5-flash-lite` vs `gemini-3.8-flash`) |
| `response["messages"][-1].usage_metadata` | input/output/total tokens per call — 61 vs 236 short, 153 vs 542 long |

---
*by Kassouar Fatima 🇩🇿 — Module 3 project*